In [ ]:
## Importing Libraries

import os

import numpy as np

import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn

from torchvision import models
import torchvision.transforms as transforms

print("Librariries Imported Successfully")

In [ ]:
## Setting Device

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device :", DEVICE)

In [ ]:
## Setting Path

PROJECT_PATH = r"Image To Biomass"

MODEL_PATH = os.path.join(PROJECT_PATH, "Models", "model.pth")

IMAGE_PATH = r"Dataset\test\ID1001187975.jpg"

In [ ]:
## Target Columns

TARGET_COLUMNS = ["Dry Clover (g)", "Dry Dead (g)", "Dry Green (g)", "Dry Total (g)", "GDM (g)"]

In [ ]:
## Image Transform

IMAGE_HEIGHT = 256
IMAGE_WIDTH = 512

transform = transforms.Compose([
    transforms.Resize((IMAGE_HEIGHT, IMAGE_WIDTH)),
    transforms.ToTensor(),

    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

In [ ]:
## Loading Model

weights = models.EfficientNet_B3_Weights.DEFAULT
model = models.efficientnet_b3(weights=weights)
in_features = model.classifier[1].in_features

model.classifier = nn.Sequential(
    nn.Dropout(0.30),
    nn.Linear(in_features,512),
    nn.ReLU(),

    nn.Dropout(0.20),
    nn.Linear(512,128),
    nn.ReLU(),

    nn.Linear(128,5)
)

model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.to(DEVICE)
model.eval()

print("Model Loaded Successfully.")

In [ ]:
## Loading Image

image = Image.open(IMAGE_PATH).convert("RGB")
plt.figure(figsize=(10, 5))
plt.imshow(image)

plt.axis("off")
plt.title("Input Image")
plt.show()

In [ ]:
## Prediction

input_tensor = transform(image)
input_tensor = input_tensor.unsqueeze(0)
input_tensor = input_tensor.to(DEVICE)

with torch.no_grad():
    prediction = model(input_tensor)

prediction = prediction.cpu().numpy().flatten()

In [ ]:
## Displaying Predictions

print("=" * 50)
print("Predicted Biomass")
print("=" * 50)

for name, value in zip(TARGET_COLUMNS, prediction):
    print(f"{name:<20}: {value:.2f}")

In [ ]:
## Plotting Prediction

plt.figure(figsize=(10, 5))
plt.bar(TARGET_COLUMNS, prediction)

plt.ylabel("Predicted Biomass")
plt.title("Predicted Biomass for Input Image")

plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
## Summary

print("=" * 50)
print("Prediction Summary")
print("=" * 50)

print("Model Used : EfficientNet-B3")
print("Weights    : model.pth")

print("Prediction Successful.")

